In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

pd.set_option("display.max_columns",50)
pd.set_option("display.width",200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "Customer_Transactions"

In [ ]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type= 'table';")
print("Table in database:", cursor.fetchall())

In [ ]:
df = pd.read_sql_query("SELECT * FROM Customer_Transactions",con)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")

In [ ]:
df.isna().sum()

In [ ]:
df.head(5)

In [ ]:
df["AssetType"].unique()

In [ ]:
# Rule 1 - AssetType drives currency
asset_to_currency = {
    "BTC": "BTC",
    "ETC": "ETC",
    "USDT": "USDT",
    "Stablecoin": "USDT",
}
mask = df["TransactionCurrency"].isna() & df["AssetType"].isin(asset_to_currency)
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 1 filled: {mask.sum():,}")  

# Rule 2 - Crypto transaction type = currency from asset (if asset present)
mask = (df["TransactionCurrency"].isna()
        & df["TransactionType"].isin(["Crypto Buy", "Crypto Sell"])
        & df["AssetType"].notna())
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 2 filled: {mask.sum():,}")

# Rule 3 Wallent only movements defaults to GBP
mask = df["TransactionCurrency"].isna() & (df["WalletActivity"]== 1)
df.loc[mask, "TransactionCurrency"] ="GBP"
print(f"Rule 3 filled {mask.sum():,}")

# Rule 4 Cross Border defaults to USD
mask = df["TransactionCurrency"].isna() & (df["TransactionType"] == "Cross-Border Payment")
df.loc[mask,"TransactionCurrency"] = "USD"
print(f"Rule 4 filled {mask.sum():,}")

# Rule 5 Everything else - GBP (home currency)
mask = df["TransactionCurrency"].isna()
df.loc[mask,"TransactionCurrency"] = "GBP"
print(f"Rule 5 filled {mask.sum():,}")
print("\nRemaining TransactionCurrency nulls:", df["TransactionCurrency"].isna().sum())



In [ ]:
currency_to_asset = {
    "GBP": "Traditional",
    "EUR": "TraditionaL",
    "USD": "Traditional",
    "BTC": "BTC",
    "ETH": " ETH",
}

mask = df["AssetType"].isna()
df.loc[mask,"AssetType"] = df.loc[mask, "TransactionCurrency"].map(currency_to_asset)

print(f"AssetType filled from currency: {mask.sum():,}")
print("Remaining AssetType nulls:", df["AssetType"].isna().sum())

# Safety net - anything still null (shouldn't be) = Traditional
df["AssetType"] = df["AssetType"].fillna("Traditional")

In [ ]:
mode_per_customer = df.groupby("CustomerID")["CustLocation"].transform(
    lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan
)

df["CustLocation"] = df["CustLocation"].fillna(mode_per_customer).fillna("Unknown")
print("Remaining CustLocation nulls:", df["CustLocation"].isna().sum())

In [ ]:
df.isna().sum()

In [ ]:
df["CustLocation"].value_counts()

In [ ]:
df.isna().sum()

In [ ]:
import os
os.makedirs("../customer_data/artifacts", exist_ok=True)

In [ ]:
df.to_csv("../customer_data/artifacts/cleaned_transaction_data.csv", index=False)